# Ru-Tokenizer

In [1]:
import tqdm
import polars as pl

from nltk import word_tokenize
from datasets import load_dataset

We'll use [`RussianNLP/coat`](https://huggingface.co/datasets/RussianNLP/coat) as a baseline dataset for making our tokenizer

In [2]:
DSET_NAME = "RussianNLP/coat"

ds = load_dataset(DSET_NAME, "binary")

train_df = pl.from_arrow(ds.data["train"].to_batches())

In [3]:
text_raw = train_df.select(pl.col("text")).to_struct()

## Build a tokenizer module

`vocab.py`

In [4]:
from enum import Enum
from typing import List, Any, Optional, Dict
from dataclasses import dataclass


@dataclass
class BaseToken:
    id: int
    repr: str
    special: bool = False


class VocabSpecialTokenReprs(str, Enum):
    SPACE = "<|space|>"
    PREFIX_ROOT = "<|prefix_root|>"
    SUFFIX_ROOT = "<|suffix_root|>"
    BOS = "<|begining_of_sequence|>"
    EOS = "<|end_of_sequece|>"
    UNK = "<|unknown|>"


class TokenVocab:
    def __init__(self):
        self._t_id = 0
        self._id2tokens = {}
        self._repr2tokens = {}

        self._spec_tokens = {}
        self._spec_tokens[VocabSpecialTokenReprs.SPACE.value] = self.new(VocabSpecialTokenReprs.SPACE.value, True)
        self._spec_tokens[VocabSpecialTokenReprs.PREFIX_ROOT.value] = self.new(VocabSpecialTokenReprs.PREFIX_ROOT.value, True)
        self._spec_tokens[VocabSpecialTokenReprs.SUFFIX_ROOT.value] = self.new(VocabSpecialTokenReprs.SUFFIX_ROOT.value, True)
        self._spec_tokens[VocabSpecialTokenReprs.BOS.value] = self.new(VocabSpecialTokenReprs.BOS.value, True)
        self._spec_tokens[VocabSpecialTokenReprs.EOS.value] = self.new(VocabSpecialTokenReprs.EOS.value, True)
        self._spec_tokens[VocabSpecialTokenReprs.UNK.value] = self.new(VocabSpecialTokenReprs.UNK.value, True)


    def _new_tid(self) -> int:
        tid = self._t_id
        self._t_id += 1
        return tid

    @property
    def tokens(self):
        return list(self._repr2tokens.values())

    def new(self, s: str, special: bool = False) -> BaseToken:
        if not isinstance(s, str):
            raise ValueError("Only strings could be registered")

        if self.get(s) is not None:
            raise ValueError(f"Token {s} already registered")

        tid = self._new_tid()
        t = BaseToken(
            id=tid,
            repr=s,
            special=special
        )
        self._id2tokens[tid] = t
        self._repr2tokens[s] = t

        if special:
            self._spec_tokens[s] = t

        return t

    def get(self, val: int | str | BaseToken= None, default: Any = None) -> BaseToken | Any:
        if isinstance(val, int):
            return self._id2tokens.get(val, default)
        elif isinstance(val, str):
            return self._repr2tokens.get(val, default)
        elif isinstance(val, BaseToken):
            if self._repr2tokens[val.repr].id != self._id2tokens[val.id].id:
                return None
            return val
        else:
            raise ValueError(f"Unexpected token identifier: {val} of type {type(val)}")
        
    def __len__(self):
        return len(self._id2tokens)


class _PrefixTree:
    def __init__(self, vocab: TokenVocab, meta: Optional[Dict[str, Any]] = None):
        self.children = {}
        self.vocab = vocab
        self.cnt = 0
        self.meta = meta or {}
        self.term = False

    def insert(self, word: List[str | BaseToken]):
        curr = self
        for sym in word:
            if sym == "":
                raise ValueError("Empty string cannot be inserted")

            t = self.vocab.get(sym, None)
            if t is None:
                t = self.vocab.new(sym)
            node = curr.children.get(t.id, None)
            if node is None:
                node = _PrefixTree(self.vocab)
                curr.children[t.id] = node

            curr.cnt += 1
            curr = node
        curr.term = True
        return None


class _NamedTree(_PrefixTree):
    def __init__(self, t: BaseToken, vocab: TokenVocab, meta: Optional[Dict[str, Any]] = None):
        super().__init__(vocab, meta)
        self.t = t


class IndexTree(_PrefixTree):
    def __init__(self, vocab, meta = None):
        super().__init__(vocab, meta)
        self.prefix_root = vocab.get(VocabSpecialTokenReprs.PREFIX_ROOT.value)
        self.suffix_root = vocab.get(VocabSpecialTokenReprs.SUFFIX_ROOT.value)
        self.children = {
            self.prefix_root.id: _NamedTree(self.prefix_root, vocab),
            self.suffix_root.id: _NamedTree(self.suffix_root, vocab),
        }
        self.frozen = False

    def insert(self, text: List[str | BaseToken], start: int, end: int):
        if self.frozen:
            raise RuntimeError("Attempted to insert into frozen index")

        if not isinstance(text, list):
            raise ValueError("Expected text to be a list of str or tokens")
        
        for i in range(start, end):
            if i == start:
                curr = self.children[self.suffix_root.id]
            else:
                curr = self.children[self.suffix_root.id]

            for j in range(i,end):
                sym = text[j]
                if sym == "":
                    raise ValueError("Empty string cannot be inserted")

                t = self.vocab.get(sym, None)
                if t is None:
                    t = self.vocab.new(sym)
                node = curr.children.get(t.id, None)
                if node is None:
                    node = _NamedTree(t, self.vocab)
                    curr.children[t.id] = node

                curr.cnt += 1
                curr = node
            curr.term = True

    def close(self):
        self.frozen = True


`builder.py`

In [5]:
import re
import heapq


class VocabBuilder:
    def __init__(self, special_tokens: List[str]):
        vocab = TokenVocab()
        for st in special_tokens:
            vocab.new(st, True)
        self.special_tokens = list(vocab._spec_tokens.keys())
        
        self.vocab = vocab
        self.itree = IndexTree(
            vocab=vocab
        )

    def preprocess(self, s: str) -> List[str | BaseToken]:
        regex = re.compile(
            "|".join(
                map(
                    lambda s: f"(?:{s})".replace("|", "\|"),
                    self.special_tokens
                )
            )
        )
        out = []
        n = len(s)
        i = 0
        SPACE = self.vocab.get(
            VocabSpecialTokenReprs.SPACE.value
        )

        def process(end):
            nonlocal i
            while i < end:
                ch = s[i]
                if ch == " ":
                    out.append(SPACE)
                else:
                    out.append(s[i])
                i += 1

        
        for match in re.finditer(regex, s):
            b, e = match.span()
            process(b)
            t_repr = s[b:e]
            t = self.vocab.get(
                t_repr, None
            )
            i = e
            if t is None:
                raise ValueError(f"Unknown but registered special token: {t_repr}")
            out.append(t)
        process(n)
        return out

    def add(self, val: str):
        r_tokens = [
            self.vocab.get(VocabSpecialTokenReprs.BOS.value),
            *self.preprocess(val),
            self.vocab.get(VocabSpecialTokenReprs.EOS.value),
        ]
        n = len(r_tokens)

        i = 0
        j = 1
        while j < n:
            el = r_tokens[j - 1]
            if isinstance(el, BaseToken) and el.special:
                self.itree.insert(r_tokens, i, j-1)
                i = j
                j = i + 1
            else:
                j += 1

    def build(self, vocab_size: int = 50) -> None:
        self.itree.close()

        max_heap = []
        root_ptree = self.itree.children[self.itree.suffix_root.id]
        for node in root_ptree.children.values():
            prefix = node.t.repr
            for child in node.children.values():
                heapq.heappush(max_heap, (-child.cnt, prefix + child.t.repr, child))

        remaining_tokens = vocab_size - len(self.vocab)
        while remaining_tokens > 0 and len(max_heap) > 0:
            _, prefix, node = heapq.heappop(max_heap)
            remaining_tokens -= 1
            _ = self.vocab.new(prefix)
            for child in node.children.values():
                heapq.heappush(max_heap, (-child.cnt, prefix + child.t.repr, child))


`tokenizer.py`

In [42]:
PTREE_TOKEN_FIELD = "token"


class PrefixTreeTokenizer(_PrefixTree):
    def __init__(self, vocab, meta = None):
        super().__init__(vocab, meta)

        for t in vocab.tokens:
            self.insert(t)

    def insert(self, val: BaseToken):
        curr = self
        word = val.repr
        for sym in word:
            if sym == "":
                raise ValueError("Empty string cannot be inserted")

            node = curr.children.get(sym, None)
            if node is None:
                node = _PrefixTree(self.vocab)
                curr.children[sym] = node

            curr.cnt += 1
            curr = node
        curr.term = True
        curr.meta[PTREE_TOKEN_FIELD] = val

    def tokenize(self, s: str):
        n = len(s)
        UNK = self.vocab.get(VocabSpecialTokenReprs.UNK.value)
        SPACE = self.vocab.get(VocabSpecialTokenReprs.SPACE.value)
        tokens = []

        i = 0
        curr = self
        last_token = UNK
        last_pos = 0
        while i <= n:
            if i == n:
                tokens.append(last_token)
                if last_pos != n - 1:
                    i = last_pos + 1
                    continue
                else:
                    break

            sym = s[i]

            node = curr.children.get(s[i], None)
            if node is None:
                if last_token.repr == UNK.repr:
                    if sym == " ":
                        tokens.append(SPACE)
                    else:
                        tokens.append(UNK)
                    i += 1
                else:
                    tokens.append(last_token)
                    i = last_pos + 1

                last_token = UNK
                curr = self
            else:
                curr = node
                ct = curr.meta.get(PTREE_TOKEN_FIELD, None)
                if ct is not None:
                    last_pos = i
                    last_token = ct
                i += 1
        return tokens

### Sandbox space

In [43]:
special_tokens = [
    "<|system|>", "<|user|>"
]
builder = VocabBuilder(special_tokens=special_tokens)


In [44]:
for sample in tqdm.tqdm(text_raw[:20000]):
    builder.add(sample["text"])

100%|██████████| 20000/20000 [00:03<00:00, 5578.84it/s]


In [45]:
builder.build(vocab_size=10000)

In [46]:
builder.vocab._repr2tokens.keys()

dict_keys(['<|space|>', '<|prefix_root|>', '<|suffix_root|>', '<|begining_of_sequence|>', '<|end_of_sequece|>', '<|unknown|>', '<|system|>', '<|user|>', 'О', 'т', 'н', 'о', 'ш', 'е', 'и', 'я', 'Г', 'а', 'п', 'с', 'э', 'р', 'м', 'к', 'ж', 'в', 'л', 'ь', 'б', '1', '9', '0', '5', 'г', 'д', 'ч', 'з', 'В', '7', '6', 'у', 'ф', 'ц', 'К', 'ы', 'й', 'ю', 'щ', ',', 'H', 'y', 'd', 'r', 'o', '-', 'Q', 'u', 'é', 'b', 'e', 'c', 'ё', 'х', 'Л', ':', 'L', 'G', '2', '3', 'Т', '(', ')', '8', 'Е', 'Д', 'П', 'М', 'И', '«', 'Э', 'Н', 'Х', 'Я', 'K', 'a', 'm', 'i', 'n', 'l', 'j', 'ú', 't', 'З', '»', '\xa0', '.', 'O', 'p', 'S', 'Р', 'Ф', 'С', 'E', 's', '"', '4', 'А', 'N', '•', 'У', 'J', 'Б', 'Щ', '%', 'B', 'h', 'P', 'k', 'Й', '!', 'R', '—', 'ъ', 'Ш', 'g', 'f', 'M', 'U', 'C', 'A', 'Ч', ';', '&', 'q', 'x', 'I', 'D', '„', 'Ц', 'V', 'Ж', 'F', 'Ю', 'X', '/', 'z', 'W', 'T', '*', '“', 'v', 'w', '\u200b', 'Δ', '−', '+', 'ó', '?', '–', "'", '=', 'Z', '…', '@', 'ā', 'ṣ', 'ș', 'ă', '{', '\\', 'Î', 'Y', '·', '#', '’', 'á'

Test tokenizer

In [47]:
from random import randint

tokenizer = PrefixTreeTokenizer(builder.vocab)
list(
    map(
        lambda el: el.repr,
        tokenizer.tokenize(
            "Я очень люблю своего найкрасивейшего котика, пусть я и падла, однако я всегда готов дать моей коте все, что у меня есть"
        )
    )
)

['Я',
 '<|space|>',
 'очен',
 'ь',
 '<|space|>',
 'люб',
 'лю',
 '<|space|>',
 'своег',
 'о',
 '<|space|>',
 'най',
 'крас',
 'иве',
 'йше',
 'го',
 '<|space|>',
 'кот',
 'ика',
 ',',
 '<|space|>',
 'пуст',
 'ь',
 '<|space|>',
 'я',
 '<|space|>',
 'и',
 '<|space|>',
 'пад',
 'ла',
 ',',
 '<|space|>',
 'однак',
 'о',
 '<|space|>',
 'я',
 '<|space|>',
 'всегд',
 'а',
 '<|space|>',
 'готов',
 '<|space|>',
 'дат',
 'ь',
 '<|space|>',
 'мое',
 'й',
 '<|space|>',
 'кот',
 'е',
 '<|space|>',
 'все',
 ',',
 '<|space|>',
 'что',
 '<|space|>',
 'у',
 '<|space|>',
 'меня',
 '<|space|>',
 'есть']